<div align="center">

# Universidad de Sevilla  
## Grado en Ingeniería del Software  
### Escuela Técnica Superior de Ingeniería Informática  
### Inteligencia Artificial (IA) – Curso 2025/2026  

<img src="../../img/portada.png" width="300"/>

---

#  Aprendizaje automático relacional  
### Primera Convocatoria – Junio 2026  

**Profesor:** Pedro Almagro Blanco

**Autores:**  
David Espina Apellaniz  
Marco Padilla Gómez  

**Fecha de entrega:** 8 de Junio de 2025  

</div>

# 04 - Random Forest

En este notebook se entrena un modelo base de **Random Forest** para la clasificación de usuarios de Twitch según `mature`.

Random Forest combina varios árboles de decisión y suele ofrecer mayor estabilidad y mejor capacidad de generalización que un único árbol.

## Objetivo metodologico

Random Forest se incluye como modelo de conjunto basado en arboles. Permite comparar si agregar muchos arboles mejora la generalizacion respecto a un unico Decision Tree.


## 1. Importación de librerías y configuración inicial

Se cargan las funciones de `src/`, las rutas principales y las herramientas de evaluación.

In [1]:
from pathlib import Path
import sys

import pandas as pd

sys.path.append("../../")

from src.data_loader import load_processed_dataset
from src.evaluation import evaluate_model, get_classification_report, save_metrics, save_model
from src.model_training import get_base_models, split_data
from src.visualization import save_confusion_matrix_plot

ROOT = Path("../../").resolve()
DATA_PROCESSED = ROOT / "data" / "processed"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
IMG_DIR = ROOT / "img"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Carga del dataset, entrenamiento y evaluación

Se entrena Random Forest con el dataset procesado y se calculan métricas como accuracy, precision, recall y f1.

In [2]:
df = load_processed_dataset(DATA_PROCESSED / "twitch_mature_features.csv")
X = df.drop(columns=["new_id", "mature"])
y = df["mature"]
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
model = get_base_models(random_state=42)["random_forest"]
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
metrics = evaluate_model(y_test, y_pred)
metrics.update({"model": "random_forest", "split": "test"})
pd.DataFrame([metrics])

,accuracy,precision,recall,f1,model,split
0,0.721505,0.565657,0.205882,0.301887,random_forest,test


## 3. Guardado de resultados

Se guardan las métricas, el modelo entrenado y la matriz de confusión.

In [3]:
print(get_classification_report(y_test, y_pred))
save_metrics(metrics, RESULTS_DIR / "rf_base_metrics.csv")
save_model(model, MODELS_DIR / "random_forest_base.joblib")
save_confusion_matrix_plot(model, X_test, y_test, IMG_DIR / "random_forest_base_cm.png")

              precision    recall  f1-score   support

       False       0.74      0.93      0.83       658
        True       0.57      0.21      0.30       272

    accuracy                           0.72       930
   macro avg       0.65      0.57      0.56       930
weighted avg       0.69      0.72      0.67       930



### Importancia de variables

La importancia de variables ayuda a identificar que informacion pesa mas en la clasificacion: metricas relacionales del grafo, atributos nativos o variables derivadas. Este analisis es util para la memoria porque conecta resultados con interpretacion del dominio.


## 4. Importancia de las variables

Se analizan las variables más relevantes para el modelo mediante `feature_importances_`.

In [4]:
importances = pd.DataFrame({"feature": X_train.columns, "importance": model.feature_importances_}).sort_values("importance", ascending=False)
importances.head(10)

,feature,importance
9,views,0.168195
3,pagerank,0.124695
8,days,0.123318
4,closeness,0.122035
5,betweenness,0.110562
2,clustering,0.100596
7,num_features,0.090687
0,degree,0.061635
1,degree_centrality,0.061461
6,community,0.036816


## 5. Conclusiones

Random Forest es uno de los modelos principales del trabajo, ya que permite capturar relaciones no lineales y comparar la importancia de las métricas relacionales extraídas del grafo.